In [11]:
from image_organization.workflow import _build_graph
from IPython.display import Markdown, display

In [12]:
import sys
from pathlib import Path

# Walk upward from the notebook's CWD until we find the project's src/ directory.
# Jupyter sets the kernel CWD to the notebook file's directory, not the launch dir.
_cwd = Path.cwd().resolve()
_root = _cwd
for _ in range(6):
    if (_root / 'src').is_dir():
        break
    _root = _root.parent

src_dir = _root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f'project root : {_root}')
print(f'src path     : {src_dir}')
print(f'exists       : {src_dir.exists()}')

project root : C:\Users\jfhar\github\desktop_organizer
src path     : C:\Users\jfhar\github\desktop_organizer\src
exists       : True


# Image Organization — LangGraph Workflow Diagrams

This notebook renders both available graph modes for the image organizer:

| Mode | `use_agent` | Description |
|------|-------------|-------------|
| **StateGraph** | `false` | Nodes call `_impl` functions directly — no LLM involvement in data flow |
| **ReAct Agent** | `true`  | LLM decides when/how to call tools; suited for reasoning tasks but may struggle with large file lists |

> **Prerequisites:** Launch Jupyter from the project root so that `src/` and `config/` resolve correctly.
> The run scripts in `script/bash/notebooks/` and `script/bat/notebooks/` do this automatically.

In [13]:
import yaml

config_path = _root / 'config' / 'config.yaml'
with open(config_path, encoding='utf-8') as f:
    config = yaml.safe_load(f)

use_agent = config.get('use_agent', False)
print(f'use_agent = {use_agent}')
print(f'Active mode: {"ReAct Agent" if use_agent else "StateGraph (direct)"}')

use_agent = False
Active mode: StateGraph (direct)


## 1 — StateGraph (direct function calls)
Nodes invoke `_scan_images_impl` and `_move_images_impl` directly.
The LLM is never in the data-flow path.

In [14]:
from image_organization.workflow import _build_graph
from IPython.display import Markdown, display

state_graph_app = _build_graph()
state_mermaid = state_graph_app.get_graph().draw_mermaid()

print(state_mermaid)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	scan(scan)
	move(move)
	__end__([<p>__end__</p>]):::last
	__start__ --> scan;
	scan --> move;
	move --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [15]:
from IPython.display import Markdown, display

display(Markdown(f'```mermaid\n{state_mermaid}\n```'))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	scan(scan)
	move(move)
	__end__([<p>__end__</p>]):::last
	__start__ --> scan;
	scan --> move;
	move --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## 2 — ReAct Agent (LLM-dispatched tool calls)
The LLM node decides which tool to call and passes arguments between them.
Tool calls appear as explicit edges in the graph.

> Building this graph instantiates `ChatOllama` but does **not** connect to Ollama —
> graph construction is purely structural.

In [16]:

sys.path.append("C:\\Users\\jfhar\\github\\desktop_organizer\\src")
sys.path.append("C:\\Users\\jfhar\\github\\desktop_organizer")
from langgraph.prebuilt import create_react_agent
from model_config import get_text_model
from image_organization.tools import scan_images, move_images

# Build agent graph for structural visualization only — no Ollama call made.
agent_app = create_react_agent(model=get_text_model(), tools=[scan_images, move_images])
agent_mermaid = agent_app.get_graph().draw_mermaid()

print(agent_mermaid)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



C:\Users\jfhar\AppData\Local\Temp\ipykernel_10936\352731377.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_app = create_react_agent(model=get_text_model(), tools=[scan_images, move_images])


In [17]:
display(Markdown(f'```mermaid\n{agent_mermaid}\n```'))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Active mode (from config.yaml)

In [18]:
active_label  = 'ReAct Agent' if use_agent else 'StateGraph (direct)'
active_mermaid = agent_mermaid if use_agent else state_mermaid

print(f'use_agent={use_agent}  →  rendering: {active_label}')
display(Markdown(f'### Currently active: {active_label}'))
display(Markdown(f'```mermaid\n{active_mermaid}\n```'))

use_agent=False  →  rendering: StateGraph (direct)


### Currently active: StateGraph (direct)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	scan(scan)
	move(move)
	__end__([<p>__end__</p>]):::last
	__start__ --> scan;
	scan --> move;
	move --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```